🟩 셀 1 — 라이브러리 임포트 및 설정

In [56]:
import os
import pymysql
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma


🟩 셀 2 — MySQL 접속 설정

In [57]:
DB_CONFIG = {
    'host': 'localhost',
    'user': 'admin',
    'password': '1qazZAQ!',
    'db': 'final',
    'charset': 'utf8mb4'
}


🟩 셀 3 — MySQL에서 문서 불러오기

In [58]:
def load_documents_from_mysql():
    conn = None
    documents = []

    try:
        conn = pymysql.connect(**DB_CONFIG)
        print("✅ MySQL 연결 성공")

        with conn.cursor(pymysql.cursors.DictCursor) as cursor:
            cursor.execute("SELECT id, file_name, title, summary FROM documents")
            rows = cursor.fetchall()

            if not rows:
                print("⚠️ documents 테이블에 데이터가 없습니다.")
                return []

            for row in rows:
                doc_id = row["id"]
                file_name = (row["file_name"] or "").strip()
                title_text = (row["title"] or "").strip()
                summary_text = (row["summary"] or "").strip()
                combined_text = f"{title_text}. {summary_text}".strip(". ")

                if not combined_text:
                    continue

                documents.append(Document(
                    page_content=combined_text,
                    metadata={
                        "source": "mysql",
                        "table": "documents",
                        "id": doc_id,
                        "file_name": file_name,
                        "title": title_text,
                        "summary": summary_text
                    }
                ))

        print(f"✅ 총 {len(documents)}개 문서 로드 완료")

        # 미리보기
        for i, doc in enumerate(documents[:5]):
            print(f"\n--- 문서 {i+1} ---")
            print(f"ID: {doc.metadata['id']}")
            print(f"제목: {doc.metadata['title']}")
            print(f"요약: {doc.metadata['summary']}")
            print(f"파일명: {doc.metadata['file_name']}")
            print(f"내용 일부: {doc.page_content[:150]}...")

    except pymysql.Error as err:
        print(f"❌ MySQL 오류: {err}")
    finally:
        if conn:
            conn.close()
            print("🔒 MySQL 연결 해제")

    return documents


🟩 셀 4 — 문서 불러오기 실행

In [59]:
documents = load_documents_from_mysql()
len(documents)

✅ MySQL 연결 성공
✅ 총 18개 문서 로드 완료

--- 문서 1 ---
ID: 597
제목: 이재명 대통령, 제77회 국군의 날 기념행사 주재
요약: 이재명 대통령이 처음으로 주관한 제77회 국군의 날 기념행사는 '국민과 함께하는 선진 강군'이라는 주제 아래 개최되었다. 이날 행사는 국민대표 7인을 포함하여 군 지휘부와 함께 진행되었으며, 헌법 수호와 민주주의 지키기의 중요성 강조와 함께 한국형 첨단 전력 체계인 K-방산 전력 공개를 통해 현대적인 군사력의 위상을 과시했다. 대통령은 장병들의 사기 진작을 목표로 국민의 지지와 애국심을 강조하는 자리로 마련되었다.
파일명: 국민의군대.docx
내용 일부: 이재명 대통령, 제77회 국군의 날 기념행사 주재. 이재명 대통령이 처음으로 주관한 제77회 국군의 날 기념행사는 '국민과 함께하는 선진 강군'이라는 주제 아래 개최되었다. 이날 행사는 국민대표 7인을 포함하여 군 지휘부와 함께 진행되었으며, 헌법 수호와 민주주의 지키...

--- 문서 2 ---
ID: 598
제목: 윤석열 전 대통령 교정수발 관련 폭로 및 법무부 감찰 착수
요약: 윤석열 전 대통령이 내란 혐의로 수감 중일 때 교정 직원 7명이 그의 개인적인 생활을 수발드렸다는 폭로가 제기되어 법무부가 감찰 조사에 착수했다. 2023년 4월, 게시글을 통해 미용사와의 만남, 변호사 접견의 부당성 등 여러 특혜 사례가 언급되었고, 이는 교정 시스템의 투명성과 책임성에 대한 심각한 의문을 제기했다. 법무부는 근무일지 부족과 같은 문제를 포함한 포괄적인 검토를 진행 중이며, 만약의 문제점이 드러나면 예산과 평가에 악영향을 미칠 수 있다고 경고했다.
파일명: 랭킹뉴스.docx
내용 일부: 윤석열 전 대통령 교정수발 관련 폭로 및 법무부 감찰 착수. 윤석열 전 대통령이 내란 혐의로 수감 중일 때 교정 직원 7명이 그의 개인적인 생활을 수발드렸다는 폭로가 제기되어 법무부가 감찰 조사에 착수했다. 2023년 4월, 게시글을 통해 미용사와의 만남, 변호사 접견..

18

🟩 셀 5 — 문서 청킹 (텍스트 분할)

In [60]:
if not documents:
    print("⚠️ 불러온 문서가 없습니다.")
else:
    splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    split_docs = splitter.split_documents(documents)
    print(f"✅ 청킹 완료: 총 {len(split_docs)}개 청크 생성")

    # 일부 확인
    for i, d in enumerate(split_docs[:3]):
        print(f"\n--- 청크 {i+1} ---")
        print(d.page_content[:200] + "...")


✅ 청킹 완료: 총 32개 청크 생성

--- 청크 1 ---
이재명 대통령, 제77회 국군의 날 기념행사 주재. 이재명 대통령이 처음으로 주관한 제77회 국군의 날 기념행사는 '국민과 함께하는 선진 강군'이라는 주제 아래 개최되었다. 이날 행사는 국민대표 7인을 포함하여 군 지휘부와 함께 진행되었으며, 헌법 수호와 민주주의 지키기의 중요성 강조와 함께 한국형 첨단 전력 체계인 K-방산 전력 공개를 통해 현대적인 군사...

--- 청크 2 ---
윤석열 전 대통령 교정수발 관련 폭로 및 법무부 감찰 착수. 윤석열 전 대통령이 내란 혐의로 수감 중일 때 교정 직원 7명이 그의 개인적인 생활을 수발드렸다는 폭로가 제기되어 법무부가 감찰 조사에 착수했다. 2023년 4월, 게시글을 통해 미용사와의 만남, 변호사 접견의 부당성 등 여러 특혜 사례가 언급되었고, 이는 교정 시스템의 투명성과 책임성에 대한 심...

--- 청크 3 ---
국가정보자원관리원 화재 원인 조사 중 업무상 실화 혐의 입건. 대전 국가정보자원관리원 화재 조사 과정에서 업무상 실화 혐의로 현장 책임자 및 작업자 등 4명이 입건되었다. 2025년 9월 29일 발생한 이 화재는 리튬이온 배터리 이전 작업 중 폭발로 인해 발생한 것으로 추정되며, 경찰은 작업 중 무정전·전원(UPS) 장치의 부적절한 관리와 잔류 전력 가능성...


🟩 셀 6 — 임베딩 모델 설정

In [61]:
try:
    # ⚠️ exaone3.5:2.4b 대신 nomic-embed-text 같은 임베딩 모델 권장
    embeddings = OllamaEmbeddings(model="exaone3.5:2.4b")
    print("✅ 임베딩 모델 준비 완료")
except Exception as e:
    print(f"❌ 임베딩 모델 설정 오류: {e}")


✅ 임베딩 모델 준비 완료


In [62]:
#ollama serve & ollama pull exaone3.5:2.4b

🟩 셀 7 — Chroma 벡터스토어 생성 및 저장

In [63]:
rag_path = "./rag_chroma/documents/final_test4/"
os.makedirs(rag_path, exist_ok=True)

print("⏳ 벡터스토어 생성 중...")

db = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory=rag_path
)
db.persist()

print(f"🎉 Chroma 벡터스토어 구축 완료!")
print(f"저장 경로: {rag_path}")


⏳ 벡터스토어 생성 중...
🎉 Chroma 벡터스토어 구축 완료!
저장 경로: ./rag_chroma/documents/final_test4/


🟩 셀 8 — 벡터스토어 확인 (테스트 검색)

In [84]:
# 저장된 벡터스토어 로드 후 테스트 질의
db_test = Chroma(persist_directory=rag_path, embedding_function=embeddings)

query = "산업통산자원부 장관"
results = db_test.similarity_search(query, k=5)

print(f"🔍 질의: {query}")
for i, r in enumerate(results):
    print(f"\n--- 결과 {i+1} ---")
    print(f"내용: {r.page_content[:200]}...")
    print(f"메타데이터: {r.metadata}")


🔍 질의: 산업통산자원부 장관

--- 결과 1 ---
내용: 한강버스 해상시운전 결과 공개, 서울시 초기 속도 미달 문제 제기. 서울시가 한강버스의 해상 시운전 과정에서 이미 초기 공언했던 평균속도(17노트)와 최대속도(20노트)를 충족시키지 못한 사실이 공개되었다. 국회의 의혹 제기로 인해 공개된 자료에 따르면, 실제로 한강버스 선박들의 평균 최고속도는 15.8노트(시속 29km)에 불과했으며, 특히 10호선은 1...
메타데이터: {'summary': '서울시가 한강버스의 해상 시운전 과정에서 이미 초기 공언했던 평균속도(17노트)와 최대속도(20노트)를 충족시키지 못한 사실이 공개되었다. 국회의 의혹 제기로 인해 공개된 자료에 따르면, 실제로 한강버스 선박들의 평균 최고속도는 15.8노트(시속 29km)에 불과했으며, 특히 10호선은 16.98노트에 그쳤다. 서울시는 이를 인지했음에도 불구하고 정식 운항 발표 직전까지 정확한 속도 정보를 수정하며 시민을 기만했다는 비판을 받고 있다. 한강버스의정식 운행 중단 이후, 선박 품질 문제와 안전 관련 논란이 심화되고 있으며, 서울시는 선박 품질 점검과 관련 조치에 나서야 할 상황이다.', 'file_name': '속도미달.docx', 'table': 'documents', 'source': 'mysql', 'title': '한강버스 해상시운전 결과 공개, 서울시 초기 속도 미달 문제 제기', 'id': 605}

--- 결과 2 ---
내용: 이재명 대통령의 국군 개혁 및 한미동맹 강화 강조 기념사. 이재명 대통령은 취임 첫 국군의 날 기념사에서 군의 민주적 기반 강화와 첨단 국방기술 투자 확대를 통해 자주국방을 실현하고, 과거 비상계엄 사태의 재발 방지를 약속했다. 특히, 국방개혁의 목표는 군을 '제복 입은 민주시민'으로 변화시키는 것으로, 이를 통해 국민의 신뢰를 높이고 국격을 향상시키려는 ...
메타데이터: {'id': 603, 'title': '이재명 대통령의 국군 개혁 및 한미동맹 강화 강조 기념사',